In [8]:
import duckdb
con = duckdb.connect('/workspaces/mlops_meia_v2/morocco_housing_ingestion.duckdb')
df = con.execute("SELECT * FROM main.stg_housing_listings").df()
df.shape
df.head()

,listing_id,source_listing_id,source,price_mad,surface_m2,price_per_sqm_calculated,rooms,bedrooms,bathrooms,floor,address,localisation,elevator,terrace,parking,property_type,city,other_tags,loaded_at
0,morocco_3,3,kaggle_morocco_housing,2600000.0,150.0,17333.333333,<NA>,3,2,3,Appartement à vendre Casablanca - Californie,Californie,1,1,1,Appartement,Casablanca,NaN,2026-08-14 18:38:13.181735+00:00
1,morocco_4,4,kaggle_morocco_housing,2290000.0,130.0,17615.384615,<NA>,2,1,3,Appartement à vendre Casablanca - Les Hôpitaux,Les Hôpitaux,1,0,0,Appartement,Casablanca,NaN,2026-08-14 18:38:13.181735+00:00
2,morocco_5,5,kaggle_morocco_housing,1460000.0,163.0,8957.055215,<NA>,3,2,2,Appartement à vendre Meknès - Hamria,Hamria,1,0,0,Appartement,Meknès,NaN,2026-08-14 18:38:13.181735+00:00
3,morocco_8,8,kaggle_morocco_housing,1600000.0,123.0,13008.130081,<NA>,3,2,4,Appartement à vendre Casablanca - Ghandi,Ghandi,1,0,0,Appartement,Casablanca,NaN,2026-08-14 18:38:13.181735+00:00
4,morocco_9,9,kaggle_morocco_housing,1140000.0,107.0,10654.205607,<NA>,2,1,3,Appartement à vendre Casablanca - Belvédère,Belvédère,1,0,0,Appartement,Casablanca,NaN,2026-08-14 18:38:13.181735+00:00


In [9]:
df.shape 

(1197, 19)

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype                  
---  ------                    --------------  -----                  
 0   listing_id                1197 non-null   str                    
 1   source_listing_id         1197 non-null   int64                  
 2   source                    1197 non-null   str                    
 3   price_mad                 1197 non-null   float64                
 4   surface_m2                1197 non-null   float64                
 5   price_per_sqm_calculated  1197 non-null   float64                
 6   rooms                     1172 non-null   Int64                  
 7   bedrooms                  1197 non-null   int64                  
 8   bathrooms                 1197 non-null   int64                  
 9   floor                     1197 non-null   int64                  
 10  address                   25 non-null     str  

In [11]:
df.isna().sum()

listing_id                     0
source_listing_id              0
source                         0
price_mad                      0
surface_m2                     0
price_per_sqm_calculated       0
rooms                         25
bedrooms                       0
bathrooms                      0
floor                          0
address                     1172
localisation                   0
elevator                    1172
terrace                     1172
parking                     1172
property_type                  0
city                           0
other_tags                    25
loaded_at                      0
dtype: int64

In [12]:
df["price_mad"].describe()

count    1.197000e+03
mean     4.890474e+06
std      7.339388e+06
min      1.600000e+03
25%      1.200000e+06
50%      2.235000e+06
75%      5.600000e+06
max      7.140000e+07
Name: price_mad, dtype: float64

## **PARTIE 1 - Nettoyer et préparer les données**


**Gérer les valeurs aberrantes de prix** : Le minimum à 1 600 MAD est clairement une erreur de saisie (personne ne vend un bien 1600 dirhams)


In [13]:
df = df[(df["price_mad"] > 50000) & (df["price_mad"] < 30000000)]
print(df.shape)

(1161, 19)


In [ ]:
# Supprimer les doublons éventuels
df = df.drop_duplicates()

(1197, 19)


**Sélectionner uniquement les colonnes utiles**

In [14]:
colonnes_a_garder = [
    "price_mad", "surface_m2", "bedrooms", "bathrooms",
    "floor", "rooms", "localisation", "property_type", "city"
]
df = df[colonnes_a_garder]
df.isna().sum()

price_mad         0
surface_m2        0
bedrooms          0
bathrooms         0
floor             0
rooms            25
localisation      0
property_type     0
city              0
dtype: int64

**Gérer les 25 valeurs manquantes de room**: Vu que c'est peu (25 sur 1197, soit ~2%), on impute avec la médiane plutôt que de supprimer des lignes

In [15]:
df["rooms"] = df["rooms"].fillna(df["rooms"].median())

In [16]:
df.isna().sum()

price_mad        0
surface_m2       0
bedrooms         0
bathrooms        0
floor            0
rooms            0
localisation     0
property_type    0
city             0
dtype: int64

## **PARTIE 2 — Feature engineering**

**Vérifier la cardinalité des colonnes catégorielles**

In [17]:
print("Villes uniques :", df["city"].nunique())
print("Types de bien uniques :", df["property_type"].nunique())
print("Localisations uniques :", df["localisation"].nunique())

Villes uniques : 7
Types de bien uniques : 8
Localisations uniques : 109


In [18]:
df["localisation"].value_counts()

localisation
Californie          114
Racine               62
Ain Diab             51
Maârif Extension     51
Bourgogne Ouest      50
                   ... 
Hay Hakam             1
Derb Salama           1
Al Osra               1
Hay Chrifa            1
Hay Arsalan           1
Name: count, Length: 109, dtype: int64

In [ ]:
seuil = 5
counts = df["localisation"].value_counts()
quartiers_rares = counts[counts < seuil].index
df["localisation_grouped"] = df["localisation"].where(~df["localisation"].isin(quartiers_rares), "Autre")
print(df["localisation_grouped"].nunique())  

49


**Séparer X (features) et y (cible)**

In [20]:
X = df.drop(columns=["price_mad", "localisation"])
y = df["price_mad"]

**Préprocesseur avec 3 colonnes catégorielles**

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

colonnes_categorielles = ["city", "property_type", "localisation_grouped"]
colonnes_numeriques = ["surface_m2", "bedrooms", "bathrooms", "floor", "rooms"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), colonnes_categorielles)
    ],
    remainder="passthrough"
)

**Split train/test**

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (928, 8) Test: (233, 8)


**Construire les 3 pipelines**

In [24]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

modeles = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "xgboost": XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
}

pipelines = {}
for nom, modele in modeles.items():
    pipelines[nom] = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", modele)
    ])

**Entraîner**

In [25]:
for nom, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    print(f"{nom} entraîné")

linear_regression entraîné
random_forest entraîné
xgboost entraîné


## **PARTIE 3 — Évaluation**

**Prédictions et métriques**

In [26]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

resultats = []
for nom, pipeline in pipelines.items():
    preds = pipeline.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    resultats.append({"modele": nom, "RMSE": rmse, "MAE": mae, "R2": r2})

resultats_df = pd.DataFrame(resultats).sort_values("RMSE")
print(resultats_df)

              modele          RMSE            MAE        R2
2            xgboost  1.539903e+06  825580.473243  0.899573
0  linear_regression  1.645188e+06  902792.585127  0.885371
1      random_forest  1.734450e+06  912236.929729  0.872595


**Sauvegarder**

In [27]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

meilleur_nom = resultats_df.iloc[0]["modele"]
meilleur_pipeline = pipelines[meilleur_nom]

joblib.dump(meilleur_pipeline, "../models/modele_prix_immobilier.pkl")
resultats_df.to_csv("../models/comparaison_modeles.csv", index=False)

print(f"Meilleur modèle sauvegardé : {meilleur_nom}")

Meilleur modèle sauvegardé : xgboost
